In [1]:
import sqlite3
import pandas as pd
import numpy as np


np.random.seed(42)


conn = sqlite3.connect('parking_system.db')
cursor = conn.cursor()


cursor.execute('''
CREATE TABLE IF NOT EXISTS floors (
    floor_id INTEGER PRIMARY KEY,
    floor_name TEXT,
    total_slots INTEGER
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS guards (
    guard_id INTEGER PRIMARY KEY,
    guard_name TEXT,
    shift TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS cars (
    car_id INTEGER PRIMARY KEY AUTOINCREMENT,
    car_type TEXT,
    entry_hour INTEGER,
    parking_duration_hrs REAL,
    hourly_rate REAL,
    floor_id INTEGER,
    guard_id INTEGER,
    is_vip INTEGER,
    FOREIGN KEY(floor_id) REFERENCES floors(floor_id),
    FOREIGN KEY(guard_id) REFERENCES guards(guard_id)
)
''')


cursor.executemany("INSERT OR IGNORE INTO floors VALUES (?, ?, ?)", [
    (1, 'Ground Floor - Zone A', 100),
    (2, 'First Floor - Zone B', 120),
    (3, 'Second Floor - Zone C', 150),
    (4, 'Rooftop - Zone D', 200)
])

cursor.executemany("INSERT OR IGNORE INTO guards VALUES (?, ?, ?)", [
    (101, 'Alice Smith', 'Morning'),
    (102, 'Bob Jones', 'Evening'),
    (103, 'Charlie Brown', 'Night'),
    (104, 'Diana Prince', 'Rotating')
])


car_types = ['Sedan', 'SUV', 'Hatchback', 'Electric', 'Truck']
data = []
for _ in range(2000):
    c_type = np.random.choice(car_types, p=[0.4, 0.3, 0.15, 0.1, 0.05])
    entry_h = int(np.random.randint(0, 24))
    duration = round(np.random.exponential(scale=3.0) + 0.5, 2)
    rate = 5.0 if c_type != 'Electric' else 3.5
    f_id = int(np.random.choice([1, 2, 3, 4]))
    g_id = int(np.random.choice([101, 102, 103, 104]))
    vip = int(np.random.choice([0, 1], p=[0.85, 0.15]))
    data.append((c_type, entry_h, duration, rate, f_id, g_id, vip))

cursor.executemany('''
INSERT INTO cars (car_type, entry_hour, parking_duration_hrs, hourly_rate, floor_id, guard_id, is_vip)
VALUES (?, ?, ?, ?, ?, ?, ?)
''', data)
conn.commit()


query = '''
SELECT
    c.car_id,
    c.car_type,
    c.entry_hour,
    c.parking_duration_hrs,
    c.hourly_rate,
    (c.parking_duration_hrs * c.hourly_rate) AS total_fee,
    c.is_vip,
    f.floor_name,
    f.total_slots,
    g.guard_name,
    g.shift
FROM cars c
JOIN floors f ON c.floor_id = f.floor_id
JOIN guards g ON c.guard_id = g.guard_id
'''

df = pd.read_sql_query(query, conn)
conn.close()
print(f"Master Dataset Shape: {df.shape}")
print(df.head())

Master Dataset Shape: (2000, 11)
   car_id car_type  entry_hour  parking_duration_hrs  hourly_rate  total_fee  \
0       1    Sedan          14                  4.45          5.0      22.25   
1       2    Sedan          10                  2.34          5.0      11.70   
2       3      SUV          20                 11.01          5.0      55.05   
3       4    Sedan          20                  3.38          5.0      16.90   
4       5    Sedan           9                  0.64          5.0       3.20   

   is_vip             floor_name  total_slots    guard_name     shift  
0       0  Ground Floor - Zone A          100   Alice Smith   Morning  
1       0  Ground Floor - Zone A          100  Diana Prince  Rotating  
2       0       Rooftop - Zone D          200     Bob Jones   Evening  
3       0   First Floor - Zone B          120     Bob Jones   Evening  
4       0       Rooftop - Zone D          200  Diana Prince  Rotating  


In [2]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np


df.loc[df.sample(n=10, random_state=42).index, 'parking_duration_hrs'] = np.nan


df['parking_duration_hrs'].fillna(df['parking_duration_hrs'].median(), inplace=True)
df['total_fee'] = df['parking_duration_hrs'] * df['hourly_rate']

features_df = df[['car_type', 'entry_hour', 'parking_duration_hrs', 'hourly_rate', 'total_fee', 'is_vip', 'floor_name', 'shift']].copy()


categorical_cols = ['car_type', 'floor_name', 'shift']
df_encoded = pd.get_dummies(features_df, columns=categorical_cols, drop_first=True)


numerical_cols = ['entry_hour', 'parking_duration_hrs', 'hourly_rate', 'total_fee']
scaler = StandardScaler()
df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])

print(f"Processed Features Shape: {df_encoded.shape}")

Processed Features Shape: (2000, 15)


/tmp/ipykernel_2561/2876672334.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['parking_duration_hrs'].fillna(df['parking_duration_hrs'].median(), inplace=True)


In [3]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

X_cluster = df_encoded[['parking_duration_hrs', 'total_fee']]

best_k = 2
best_score = -1
for k in range(2, 6):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_cluster)
    score = silhouette_score(X_cluster, labels)
    if score > best_score:
        best_score = score
        best_k = k

print(f"Optimal K for K-Means: {best_k} with Silhouette Score: {best_score:.4f}")


kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_cluster)


df_encoded['cluster_label'] = cluster_labels

Optimal K for K-Means: 2 with Silhouette Score: 0.6751


In [4]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier


X = df_encoded.drop(columns=['cluster_label'])
y = df_encoded['cluster_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)


param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5],
    'learning_rate': [0.1, 0.2]
}
grid_search = GridSearchCV(GradientBoostingClassifier(random_state=42), param_grid, cv=3, scoring='accuracy')
grid_search.fit(X_train, y_train)

best_classifier = grid_search.best_estimator_
print(f"Best Hyperparameters for Gradient Boosting: {grid_search.best_params_}")

Best Hyperparameters for Gradient Boosting: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}


In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_pred = best_classifier.predict(X_test)

print("--- Model Evaluation Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred, average='weighted'):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

--- Model Evaluation Metrics ---
Accuracy:  0.9975
Precision: 0.9975
Recall:    0.9975
F1-Score:  0.9975

Confusion Matrix:
[[ 83   0]
 [  1 316]]


In [6]:
import joblib

# Bundle pipeline artifacts
pipeline_artifact = {
    'model': best_classifier,
    'scaler': scaler,
    'feature_columns': X.columns.tolist()
}

# Save
joblib.dump(pipeline_artifact, 'parking_ml_pipeline.joblib')

# Load
loaded_artifact = joblib.load('parking_ml_pipeline.joblib')
loaded_model = loaded_artifact['model']

# Verify consistency
original_preds = best_classifier.predict(X_test[:5])
reloaded_preds = loaded_model.predict(X_test[:5])
assert np.all(original_preds == reloaded_preds), "Model serialization verification failed!"
print("Model successfully saved, reloaded, and verified!")

Model successfully saved, reloaded, and verified!


In [7]:
from pydantic import BaseModel, Field, field_validator

class ParkingInputSchema(BaseModel):
    car_type: str = Field(..., description="Type of car (Sedan, SUV, Hatchback, Electric, Truck)")
    entry_hour: int = Field(..., ge=0, le=23, description="Hour of entry (0-23)")
    parking_duration_hrs: float = Field(..., gt=0, description="Duration in hours")
    hourly_rate: float = Field(..., gt=0, description="Rate per hour")
    is_vip: int = Field(..., ge=0, le=1, description="1 if VIP, 0 otherwise")
    floor_name: str = Field(..., description="Floor location name")
    shift: str = Field(..., description="Guard shift")

    @field_validator('car_type')
    def validate_car_type(cls, v):
        allowed = ['Sedan', 'SUV', 'Hatchback', 'Electric', 'Truck']
        if v not in allowed:
            raise ValueError(f"Invalid car type. Must be one of {allowed}")
        return v

# Test validation
sample_input = {
    "car_type": "SUV",
    "entry_hour": 14,
    "parking_duration_hrs": 2.5,
    "hourly_rate": 5.0,
    "is_vip": 1,
    "floor_name": "First Floor - Zone B",
    "shift": "Morning"
}
validated_data = ParkingInputSchema(**sample_input)
print("Input validation passed successfully!")

Input validation passed successfully!


Model Card
 Parking Behavior Classification Model Card
Overview: Predicts parking behavior segments (short-term vs. long-term/high-fee) for optimized parking allocation in a 4-floor facility.

Intended Use: Automated parking tier classification, dynamic pricing adjustments, and security shift allocation.

Training Data: 2,000 synthetic relational records synthesized from SQL tables (cars, floors, guards).

Algorithms Used:

Unsupervised: K-Means (Cluster generation)

Supervised: Gradient Boosting Classifier

Evaluation Results: Achieved >95% accuracy and robust F1-score across clustered target classes.

Limitations: Assumes linear behavioral trends based on duration and rates; sensitive to extreme outliers in parking duration.

Potential Risks: Over-reliance on synthetic training distributions may lead to classification bias during peak holiday surges.